In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Let's start with necessary imports
import os
import numpy as np
from shutil import copyfile
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt

## Define Settings

In [3]:
data_dir = "../../data/images/"
meta_data_path = "../../data/butterfly_anomaly_train.csv"
plot_dir = "../../plots/"
model_dir = "../../data/models/"

In [4]:
for dir_name in [plot_dir, model_dir]:
    if not os.path.exists(dir_name):
        os.makedirs(dir_name)

## Define Data Loader

In [ ]:
from hdr_hybrid_butterflies.data_handler import DataHandler, ImageProcessor

data_handler = DataHandler(
    meta_data_path=meta_data_path,
    data_dir=data_dir,
)

In [ ]:
data_handler.df_meta.iloc[10]["hybrid_stat"] == "nonhybrid"

In [ ]:
data_handler.df_meta.iloc[10]

In [ ]:
data_handler.load_data(1804)[0]

### Copy Files of Subspecies to sub-directories

In [ ]:
np.unique(data_handler.df_meta["subspecies"], return_counts=True)

In [10]:
if False:
    for idx, row in tqdm(
        data_handler.df_meta.iterrows(), total=data_handler.n_samples
    ):
        if row["hybrid_stat"] == "non-hybrid":
            input_path = os.path.join(
                data_dir, row["hybrid_stat"], row["filename"]
            )
            output_path = os.path.join(
                data_dir,
                row["hybrid_stat"],
                f"{int(row['subspecies']):02d}",
                row["filename"],
            )
            output_dir = os.path.dirname(output_path)
            if not os.path.exists(output_dir):
                os.makedirs(output_dir)
            copyfile(input_path, output_path)

## Create overview of all Subspecies

In [ ]:
n_subspecies = len(np.unique(data_handler.df_meta["subspecies"]))
n_subspecies

In [ ]:
data_handler.df_meta[
    data_handler.df_meta["subspecies"].isnull()
].parent_subspecies_2.unique()

In [13]:
if False:
    for seed in range(10):
        rng = np.random.default_rng(seed)

        sub_species = np.unique(data_handler.df_meta["subspecies"])

        fig, axes = plt.subplots(3, 5, figsize=(30, 15))
        axes_flat = axes.flatten()

        for idx, sub in enumerate(sub_species):
            if np.isnan(sub):
                indices = data_handler.df_meta[
                    data_handler.df_meta["subspecies"].isnull()
                ].index
            else:
                indices = data_handler.df_meta[
                    data_handler.df_meta["subspecies"] == sub
                ].index

            chosen_idx = rng.choice(indices)

            img, row = data_handler.load_data(chosen_idx)
            axes_flat[idx].imshow(img)
            axes_flat[idx].set_title(
                f"Subspecies: {row['subspecies']} | Idx: {chosen_idx} | Occurrences: {len(indices)}"
            )
            axes_flat[idx].axis("off")

        plt.tight_layout()
        fig.savefig(
            os.path.join(plot_dir, f"subspecies_examples_{seed:04d}.png")
        )

## Test Grounded Segment Anything (Grounded DINO + SAM)

In [ ]:
image, row = data_handler.load_data(874)
image

In [ ]:
from PIL import ImageOps

# image, row = data_handler.load_by_name("CAM008547.jpg")
image, row = data_handler.load_by_name("CAM011441")
# image, row = data_handler.load_by_name("CAM000446")
image = ImageOps.exif_transpose(image)
image

In [16]:
from hdr_hybrid_butterflies.dino_sam.dino_sam import grounded_segmentation

In [ ]:
1237
# labels = ["upper left wing.", "lower left wing.", "upper right wing.", "lower right wing."]
labels = [
    "upper left butterfly wing.",
    "lower left butterfly wing.",
    "upper right butterfly wing.",
    "lower right butterfly wing.",
]
# labels = ["upper wing.", "lower wing."]
# labels = ["left wing.", "right wing."]
labels = ["wings."]
# labels = ["upper wing.", "lower wing."]
# labels = ["larger wing.", "smaller wing."]
# labels = ["butterfly wing."]
threshold = 0.2

detector_id = "IDEA-Research/grounding-dino-tiny"
segmenter_id = "facebook/sam-vit-base"


image_array, detections = grounded_segmentation(
    image=image,
    labels=labels,
    threshold=threshold,
    polygon_refinement=True,
    detector_id=detector_id,
    segmenter_id=segmenter_id,
)

In [18]:
from hdr_hybrid_butterflies.dino_sam import plotting

In [ ]:
def get_most_confident_detection(detections, label):
    detections_mask = [d for d in detections if d.label == label]
    return sorted(detections_mask, key=lambda x: x.score)[-1]


detections_confident = [
    get_most_confident_detection(detections, label) for label in labels
]
detections_confident

In [ ]:
fig, ax = plotting.plot_detections(image_array, detections)
fig.savefig(os.path.join(plot_dir, "detections.png"))

In [ ]:
fig, ax = plotting.plot_detections(image_array, detections_confident)

## Create training data for segment classifier 

Classified created image segments into one of the following classes: 
    upper / lower / noise
Further steps will only be run on segments that pass the classifier

In [22]:
segment_training_dir = os.path.join(data_dir, "segment_classier_training")

if not os.path.exists(segment_training_dir):
    os.makedirs(segment_training_dir)

In [ ]:
from PIL import Image
from glob import glob

data_processor = ImageProcessor()

for idx, row in tqdm(
    data_handler.df_meta.iterrows(), total=data_handler.n_samples
):
    file_glob = f"{row['filename'][:-4]}_*.jpg"
    if len(glob(os.path.join(segment_training_dir, "all", file_glob))) > 0:
        print(f"Skipping {row['filename']}")
        continue

    image, row = data_handler.load_data(idx)
    segments, scores = data_processor._raw_segments(image)

    for idx_j, segment in enumerate(segments):
        score = scores[idx_j]
        filename = f"{row['filename'][:-4]}_{idx_j:04d}_s{score:0.4f}.jpg"
        PIL_image = Image.fromarray(segment)
        PIL_image.save(os.path.join(segment_training_dir, "all", filename))

In [32]:
upper_wing_dir = os.path.join(segment_training_dir, "upper_wing")
lower_wing_dir = os.path.join(segment_training_dir, "lower_wing")
noise_dir = os.path.join(segment_training_dir, "noise")

for output_dir in [upper_wing_dir, lower_wing_dir, noise_dir]:
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

In [33]:
import imagesize

if False:
    segment_files = sorted(
        glob(os.path.join(segment_training_dir, "all", "*.jpg"))
    )

    for idx, filename in enumerate(segment_files):
        width, height = imagesize.get(filename)
        ratio = width / height
        score = float(filename.split("_")[-1][1:-4])
        if score > 0.5:
            if ratio > 1.4:
                copyfile(
                    filename,
                    os.path.join(upper_wing_dir, os.path.basename(filename)),
                )
            else:
                copyfile(
                    filename,
                    os.path.join(lower_wing_dir, os.path.basename(filename)),
                )
        elif score > 0.1:
            copyfile(
                filename, os.path.join(noise_dir, os.path.basename(filename))
            )

        print(f"{idx:04d} | {ratio:0.3f} | {width}x{height} | {score:0.4f}")

    len(segment_files)

In [ ]:
from hdr_hybrid_butterflies.data_handler import (
    SegmentDataHandler,
    ImageProcessor,
)

image_processor = ImageProcessor()

segment_data_handler = SegmentDataHandler(
    data_dir_upper=os.path.join(
        segment_training_dir, "manual", "upper_wing_manual"
    ),
    data_dir_lower=os.path.join(
        segment_training_dir, "manual", "lower_wing_manual"
    ),
    data_dir_noise=os.path.join(
        segment_training_dir, "manual", "noise_manual"
    ),
    image_processor=image_processor,
)
segment, segment_info = segment_data_handler.load_data(0)
segment

In [51]:
segment_generator_test = segment_data_handler.get_generator(
    batch_size=16,
    queue_size=320,
    n_jobs=1,
    mask_only=True,
    training=False,
)

In [52]:
segment, segment_label = next(segment_generator_test)
fig, axes = plt.subplots(4, 4, figsize=(20, 20))
axes_flat = axes.flatten()
for idx, ax in enumerate(axes_flat):
    ax.imshow(segment[idx])
    ax.set_title(f"Label: {segment_label[idx]}")
    ax.axis("off")

In [ ]:
%timeit segment_data_handler()

## Train Segment Classifier

In [42]:
segment_generator_train = segment_data_handler.get_generator(
    batch_size=16,
    queue_size=10000,
    n_jobs=12,
    mask_only=True,
    training=True,
)

In [ ]:
image_processor.output_dim

In [ ]:
import tensorflow as tf
from hdr_hybrid_butterflies.model import CNNClasifier

model = CNNClasifier(
    image_size=image_processor.output_dim,
    num_classes=3,
)
print(model.call(segment).shape)
model.summary()

In [55]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)

In [56]:
segment_model_dir = os.path.join(model_dir, "segment_model")

In [ ]:
model.load_weights(os.path.join(segment_model_dir, "model.weights.h5"))

In [58]:
# Create the ModelCheckpoint callback
checkpoint_path = os.path.join(segment_model_dir, "model.weights.h5")

model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=True,
    monitor="loss",  # 'val_loss'
    mode="min",
    save_best_only=False,
    save_freq="epoch",  # Save every epoch
    verbose=1,
)

In [ ]:
steps_per_epoch = 100
epochs = 10


model.fit(
    segment_generator_train,
    validation_data=segment_generator_test,
    steps_per_epoch=steps_per_epoch,
    epochs=epochs,
    validation_steps=5,
    callbacks=[model_checkpoint_callback],
)

#### Evaluate Segmentation Model

In [ ]:
segments, segment_labels = next(segment_generator_test)

# get prediction
logits = model(segments)
probs = model.logits2probs(logits)

fig, axes = plt.subplots(4, 4, figsize=(20, 20))
axes_flat = axes.flatten()
for idx, ax in enumerate(axes_flat):
    ax.imshow(segment[idx])
    ax.set_title(
        f"Label: {segment_label[idx]} | Pred: {np.argmax(probs[idx])} ["
        f"{probs[idx][0]:0.2f}, {probs[idx][1]:0.2f}, {probs[idx][2]:0.2f}]"
    )
    ax.axis("off")

In [76]:
%matplotlib inline